In [ ]:
# =========================================================
# COMPONENT 1 — LANGUAGE MODEL
# Dataset File: product_reviews.csv
# =========================================================

import pandas as pd
import re
import math
import nltk

from collections import Counter, defaultdict

from sklearn.model_selection import train_test_split

from nltk.corpus import stopwords

# =========================================================
# DOWNLOAD NLTK DATA
# =========================================================

nltk.download('stopwords')

# =========================================================
# STOPWORDS
# =========================================================

stop_words = set(stopwords.words('english'))

# =========================================================
# LOAD DATASET
# =========================================================

df = pd.read_csv("balanced_sentiment_dataset_900.csv")

print("\nDATASET LOADED SUCCESSFULLY")

print(df.head())

# Automatically detect text column
text_col = df.columns[0]

print("\nTEXT COLUMN DETECTED:", text_col)

# =========================================================
# PREPROCESSING WITH NEGATION HANDLING
# =========================================================

def preprocess(text):

    text = str(text).lower()

    text = re.sub(r'[^a-z\s]', '', text)

    words = text.split()

    processed = []

    negation_words = [
        "not",
        "no",
        "never",
        "dont",
        "didnt",
        "isnt",
        "wasnt",
        "cant",
        "hardly",
        "rarely",
        "barely"
    ]

    skip_next = False

    for i in range(len(words)):

        if skip_next:
            skip_next = False
            continue

        word = words[i]

        if word in negation_words and i + 1 < len(words):

            processed.append(
                "NOT_" + words[i + 1]
            )

            skip_next = True

        else:

            if word not in stop_words:

                processed.append(word)

    return processed

# =========================================================
# TOKENIZATION
# =========================================================

df['tokens'] = df[text_col].apply(preprocess)

print("\nTOKENIZED OUTPUT")

print(df['tokens'].head())

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

train, test = train_test_split(
    df['tokens'],
    test_size=0.2,
    random_state=42
)

print("\nTRAIN SIZE:", len(train))
print("TEST SIZE:", len(test))

# =========================================================
# UNIGRAM MODEL
# =========================================================

unigram_counts = Counter()

for sentence in train:

    unigram_counts.update(sentence)

print("\nUNIGRAM SAMPLE")

print(dict(list(unigram_counts.items())[:10]))

# =========================================================
# BIGRAM MODEL
# =========================================================

bigram_counts = defaultdict(Counter)

for sentence in train:

    for i in range(len(sentence)-1):

        w1 = sentence[i]
        w2 = sentence[i+1]

        bigram_counts[w1][w2] += 1

print("\nBIGRAM SAMPLE")

for word in list(bigram_counts.keys())[:3]:

    print(word, dict(bigram_counts[word]))

# =========================================================
# BIGRAM PROBABILITY
# =========================================================

def bigram_probability(w1, w2):

    if unigram_counts[w1] == 0:
        return 0

    return bigram_counts[w1][w2] / unigram_counts[w1]

print("\nBIGRAM PROBABILITY")

print(
    "P(product | good) =",
    bigram_probability("good", "product")
)

# =========================================================
# ADD-ONE SMOOTHING
# =========================================================

vocab_size = len(unigram_counts)

def smoothed_bigram_probability(w1, w2):

    return (
        bigram_counts[w1][w2] + 1
    ) / (
        unigram_counts[w1] + vocab_size
    )

print("\nSMOOTHED BIGRAM PROBABILITY")

print(
    smoothed_bigram_probability("good", "product")
)

# =========================================================
# PERPLEXITY
# =========================================================

def perplexity(test_sentences):

    log_prob = 0

    N = 0

    for sentence in test_sentences:

        for i in range(len(sentence)-1):

            prob = smoothed_bigram_probability(
                sentence[i],
                sentence[i+1]
            )

            log_prob += math.log(prob)

            N += 1

    return math.exp(-log_prob / N)

pp = perplexity(test)

print("\nPERPLEXITY =", round(pp, 2))

# =========================================================
# DOMAIN CHALLENGES
# =========================================================

print("\nDOMAIN CHALLENGES")

print("1. Mixed sentiment reviews")
print("2. Informal spellings")
print("3. Emoji and abbreviation handling")